# Transform Your Documents into AI-Ready Data with Docling

## Lab 1: Document Conversion (Colab + Replicate Edition)

Welcome to the first lab in our comprehensive Docling workshop series! This three-part journey will take you from document processing basics to building cutting-edge, transparent AI systems.

**This notebook is optimized for Google Colab** and uses [Replicate](https://replicate.com) for hosted model inference, eliminating the need for local GPU resources or Ollama.

## Overview: What You'll Build

In this first lab, you'll learn how to transform complex documents into AI-ready structured data. But this isn't just about extraction - it's about intelligent conversion that preserves everything important for downstream AI applications.

### By the End of This Lab

You'll have mastered:
- **Document Loading**: Ingesting PDFs, Word docs, PowerPoints, and more
- **Structure Extraction**: Preserving hierarchy and relationships
- **Table Excellence**: Converting complex tables into usable formats
- **Image Handling**: Extracting and preparing images for AI processing
- **Metadata Preservation**: Maintaining information needed for visual grounding

---

## Introduction

**[Docling](https://docling-project.github.io/docling/)** is an open-source toolkit for document processing, parsing, and conversion designed for generative AI applications.

### Key Features
- **Multi-format support**: PDF, DOCX, XLSX, HTML, images, and more
- **Advanced PDF understanding**: Page layout, reading order, table structure, code blocks, formulas
- **Unified DoclingDocument representation**: Consistent data structure across all formats
- **Flexible export options**: Markdown, HTML, JSON, DocTags
- **Local execution**: Process sensitive data without external services
- **Framework integration**: LangChain, LlamaIndex, and other AI frameworks

## Why we use Docling for Document Conversion

Data is the foundation for all AI systems. In order to leverage as much data as possible, we need to be able to ingest data of various formats with accuracy. However, LLMs typically require data in a specific format, thus the need for conversion.

**Without proper conversion**:
- Information gets lost or jumbled
- Tables become unreadable text
- Images disappear entirely
- Document structure is destroyed

**With Docling's advanced conversion**:
- Every piece of information is preserved
- Tables maintain their structure
- Images are extracted and processable
- Layout and relationships are understood

## Installation and Setup

### Setting up the environment

Ensure you are running Python 3.10 or later in a freshly created virtual environment.

In [ ]:
import sys
assert sys.version_info >= (3, 10) and sys.version_info < (3, 14), "Use Python 3.10, 3.11, 3.12, or 3.13 to run this notebook."

### Basic Installation

In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "docling[vlm]" docling matplotlib pillow pandas ipywidgets python-dotenv \
    "langchain_replicate @ git+https://github.com/ibm-granite-community/langchain-replicate.git" \
    "git+https://github.com/ibm-granite-community/utils.git" \
    transformers
! echo "::endgroup::"

### Replicate API Setup

This notebook uses [Replicate](https://replicate.com) for hosted model inference. You'll need a Replicate API token.

**To get a token:**
1. Sign up at [replicate.com](https://replicate.com)
2. Go to [replicate.com/account/api-tokens](https://replicate.com/account/api-tokens)
3. Create a new token

**In Google Colab**, add your token to the Secrets manager (key icon in the left sidebar) with the name `REPLICATE_API_TOKEN`.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["REPLICATE_API_TOKEN"] = userdata.get("REPLICATE_API_TOKEN")
    print("Replicate API token loaded from Colab Secrets.")
except (ImportError, Exception):
    if "REPLICATE_API_TOKEN" not in os.environ:
        raise ValueError(
            "REPLICATE_API_TOKEN not found. "
            "Set it via Colab Secrets or as an environment variable."
        )
    print("Replicate API token loaded from environment.")

### Import Essential Components


In [ ]:
from pathlib import Path

# Core Docling imports
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import PdfFormatOption

# For advanced features
from docling_core.types.doc import ImageRefMode, PictureItem, TableItem, TextItem, DoclingDocument

# For data processing and visualization
import matplotlib.pyplot as plt

# Create output directory
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

## Basic Document Conversion

### Minimal Example

The simplest way to convert a document:

### Your turn: Try your own document

The cell below uses the Docling technical report as the input. **Try changing the URL** to a different PDF — a paper you've been meaning to read, a company datasheet, any PDF on the web — and re-run this cell plus the next two to see Docling convert it.

<details>
<summary>What should I try?</summary>

Some URLs to try:
- `"https://arxiv.org/pdf/2408.09869"` — the original Docling paper (lots of figures)
- `"https://arxiv.org/pdf/2502.09927"` — a paper dense with tables
- Any PDF URL you have on hand

You can also point at a **local file**: `docling_paper = "path/to/my_document.pdf"`. Docling handles `.docx`, `.html`, `.pptx`, and more — not just PDFs.

</details>

In [ ]:
# TODO: try changing the URL below to a different PDF or a local file path.
# example document: Docling Technical Report
docling_paper = "https://arxiv.org/pdf/2501.17887"  # <-- change me

In [ ]:
# Simple conversion

# Create a converter instance
converter = DocumentConverter()

# Convert a document
result = converter.convert(docling_paper)
doc = result.document

In [ ]:
# Export to Markdown
md_out = doc.export_to_markdown()

# printing an excerpt
print(f"{md_out[:2000]}...")

### Document Structure Exploration

One of Docling's superpowers is understanding document structure. This becomes critical in Lab 2 (chunking) and Lab 3 (visual grounding).

Let's explore what Docling discovers:


In [ ]:
# Document metadata - important for tracking
print(f"Document title: {doc.name}")
print(f"Number of pages: {len(doc.pages)}")
print(f"Number of tables: {len(doc.tables)}")
print(f"Number of pictures: {len(doc.pictures)}")

# Explore the document structure
print("\nDocument structure:")
for i, (item, level) in enumerate(doc.iterate_items()):
    if i < 10:  # Show first 10 items
        item_type = type(item).__name__
        text_preview = item.text[:200] if hasattr(item, 'text') else 'N/A'
        print(f"{'  ' * level}{item_type}: {text_preview}")

## Export Formats and Options

Docling supports multiple export formats with various options:


In [ ]:
# Export to different formats (various options available, but called with default ones)
markdown_text = doc.export_to_markdown()
html_text = doc.export_to_html()
json_dict = doc.export_to_dict()
doc_tags = doc.export_to_doctags()

# Save different formats (various options available, some shown)
doc.save_as_markdown(
    output_dir / "document.md",
    image_mode=ImageRefMode.PLACEHOLDER,
    image_placeholder="<!-- my image placeholder -->",
    # ...
)

# ...

# Export as JSON
doc.save_as_json(
    output_dir / "document.json",
)

## Working with Tables

Docling provides excellent table extraction capabilities, for this example, let's use a document with more tables, found [here](https://midwestfoodbank.org/images/AR_2020_WEB2.pdf)

### Basic Table Export

In [ ]:
# example document with tables
tables_example = "https://arxiv.org/pdf/2502.09927"

In [ ]:
# Convert a document with tables
table_result = converter.convert(tables_example)
table_doc = table_result.document

In [ ]:
print(f"\nDocument contains {len(table_doc.tables)} tables")

# Export all tables
for table_idx, table in enumerate(table_doc.tables):
    # Convert to pandas DataFrame
    df = table.export_to_dataframe()

    print(f"\n## Table {table_idx}")
    print(f"Shape: {df.shape}")
    display(df.head())

    # Save as CSV
    df.to_csv(output_dir / f"table_{table_idx}.csv", index=False)

    # Save as HTML
    with open(output_dir / f"table_{table_idx}.html", "w") as fp:
        fp.write(table.export_to_html(doc=table_doc))

## Image and Figure Extraction

Docling also provides image extraction capabilities

### Extracting and Visualizing Page Images

### Your turn: Image resolution

The cell below extracts page images and figures. `IMAGE_RESOLUTION_SCALE` controls how sharp those extracted images are. **Try a different value** and inspect the saved images in `output/images/` to compare.

<details>
<summary>What should I try?</summary>

`IMAGE_RESOLUTION_SCALE` multiplies the base 72 DPI. Try:
- `1.0` — base (72 DPI), smallest files, fuzziest images
- `2.0` — the default (144 DPI), balanced
- `4.0` — sharp (288 DPI), larger files, slower to generate

Higher scales produce crisper images — better for downstream vision models — but cost memory and processing time.

</details>

In [ ]:
# TODO: try changing IMAGE_RESOLUTION_SCALE below to see how extracted image quality differs.
# Configure pipeline to extract images
IMAGE_RESOLUTION_SCALE = 2.0  # <-- change me (2.0 = 144 DPI)

pipeline_options = PdfPipelineOptions(
    images_scale=IMAGE_RESOLUTION_SCALE,
    generate_page_images=True,
    generate_picture_images=True,
)

converter_with_images = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

# Convert with image extraction
img_result = converter_with_images.convert(docling_paper)
img_doc = img_result.document

In [ ]:
# Create images subdirectory
images_dir = output_dir / "images"
images_dir.mkdir(exist_ok=True)

# Save page images
for page_no, page in img_doc.pages.items():
    page_image_filename = images_dir / f"page_{page_no}.png"
    with page_image_filename.open("wb") as fp:
        page.image.pil_image.save(fp, format="PNG")

print(f"Saved {len(img_doc.pages)} page images")

### Extract Figures and Tables as Images

For custom processing, we can also extract figures and tables as images:

In [ ]:
# Extract all figures and tables as separate images
table_counter = 0
picture_counter = 0

for element, level in img_doc.iterate_items():
    if isinstance(element, TableItem):
        table_counter += 1
        image_filename = images_dir / f"table_{table_counter}.png"
        with image_filename.open("wb") as fp:
            element.get_image(img_doc).save(fp, "PNG")

    elif isinstance(element, PictureItem):
        picture_counter += 1
        image_filename = images_dir / f"figure_{picture_counter}.png"
        with image_filename.open("wb") as fp:
            element.get_image(img_doc).save(fp, "PNG")

print(f"Extracted {table_counter} tables and {picture_counter} figures as images")

### Inspecting Picture Content

Docling will automatically generate captions and extract text content from extracted images. Let's take a look at what is extracted:

In [ ]:
def inspect_pictures_with_images(doc: DoclingDocument, image_size=(6, 4)):
    """Display pictures inline with their text content"""
    for idx, picture in enumerate(doc.pictures):
        print(f"\n{'='*60}")
        print(f"Picture {idx}")
        print(f"{'='*60}")

        # Display the image
        try:
            img = picture.get_image(doc)
            if img:
                plt.figure(figsize=image_size)
                plt.imshow(img)
                plt.axis('off')
                plt.title(f"Picture {idx}")
                plt.show()
        except Exception as e:
            print(f"Could not display image: {e}")

        # Display metadata
        caption = picture.caption_text(doc)
        if caption:
            print(f"\nCaption: {caption}")

        if hasattr(picture, 'prov') and picture.prov:
            print(f"Location: Page {picture.prov[0].page_no}")

        # Display embedded text
        print("\nEmbedded text elements:")
        text_found = False
        for item, level in doc.iterate_items(root=picture, traverse_pictures=True):
            if isinstance(item, TextItem):
                print(f"{'  ' * (level + 1)}- {item.text}")
                text_found = True

        if not text_found:
            print("  (No text elements found)")

# Use the simple inline display
inspect_pictures_with_images(img_doc)

### Visualizing Document Layout with Bounding Boxes

In order to understand how each part of the document is extracted, let's visualize the extracted elements.

We can do that by using one of Docling's built-in visualizers:

In [ ]:
from docling_core.transforms.visualizer.layout_visualizer import LayoutVisualizer

layout_visualizer = LayoutVisualizer()
page_images = layout_visualizer.get_visualization(doc=img_doc)

num_pages_to_viz = 2  # first N pages to visualize
pages_to_viz = list(page_images.keys())[:num_pages_to_viz]
for page in pages_to_viz:
    display(page_images[page])

## Advanced Features: Enrichment

### Picture Description with Replicate

Beyond basic captioning, Docling can process images using multimodal LLMs for more detailed descriptions. In this Colab-optimized notebook, we use **Replicate** to host the vision model remotely, so no local GPU is required.

We use the `langchain_replicate` package to call the Granite Vision model hosted on Replicate. After converting the document with image extraction enabled, we iterate through each picture and send it to the vision model for a detailed description.

In [ ]:
from IPython.display import HTML

def display_enriched_document(enr_doc: DoclingDocument, model_name: str = "VLM"):
    html_buffer = []
    # display the first 5 pictures and their captions and annotations:
    for pic in enr_doc.pictures[:5]:
        html_item = (
            f"<h3>Picture <code>{pic.self_ref}</code></h3>"
            f'<img src="{pic.image.uri!s}" /><br />'
            f"<h4>Caption</h4>{pic.caption_text(doc=enr_doc)}<br />"
        )
        # Get VLM description from pic.meta.description
        if pic.meta and pic.meta.description:
            # Use provided model_name, or created_by if it's not "not-implemented"
            source = model_name if pic.meta.description.created_by == "not-implemented" else pic.meta.description.created_by
            html_item += (
                f"<h4>Description ({source})</h4>"
                f"{pic.meta.description.text}<br />\n"
            )
        html_buffer.append(html_item)
    display(HTML("<hr />".join(html_buffer)))

### Enrichment via Replicate API

We first convert the document with image extraction enabled, then iterate through each picture and call the Granite Vision model on Replicate to generate a description. This is the same approach used in Lab 3 (RAG).

### Your turn: Prompt the vision model

The cell below sends each picture in the document to a vision-language model (Granite Vision, hosted on Replicate) for a detailed description. The `image_prompt` tells the model what to look for. **Try changing it** and see how dramatically the descriptions change.

<details>
<summary>What should I try?</summary>

Some prompts to try:
- `"Describe this image in one sentence."` — forces a concise description
- `"Extract any text you can read in this image."` — OCR-style
- `"What type of chart is this, and what does it show?"` — targeted at figures and charts
- `"List the main objects and their relationships."` — structural

Prompt engineering applies to images too — the model follows your instructions closely.

</details>

In [ ]:
import os
import io
import base64
import PIL.ImageOps
from langchain_replicate import Replicate
from langchain_core.messages import HumanMessage
from transformers import AutoProcessor
from ibm_granite_community.notebook_utils import get_env_var
from ibm_granite_community.langchain.prompts import TokenizerChatPromptTemplate

def encode_image(image, fmt="png"):
    """Encode a PIL image to a data URI for the vision model."""
    image = PIL.ImageOps.exif_transpose(image) or image
    image = image.convert("RGB")
    buffer = io.BytesIO()
    image.save(buffer, format=fmt)
    encoding = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/{fmt};base64,{encoding}"

# Load the Granite Vision model via Replicate
vision_model_path = "ibm-granite/granite-vision-3.3-2b"
vision_model = Replicate(
    model=vision_model_path,
    replicate_api_token=get_env_var("REPLICATE_API_TOKEN"),
    model_kwargs={
        "max_tokens": 200,
        "temperature": 0.01,
    },
)
vision_processor = AutoProcessor.from_pretrained(vision_model_path)

# Configure vision prompt
# TODO: try changing image_prompt below to ask a different question about each image.
image_prompt = "Give a detailed description of what is depicted in the image"  # <-- change me
vision_prompt_template = TokenizerChatPromptTemplate.from_messages(
    messages=[
        HumanMessage(content=[
            {"type": "image"},
            {"type": "text", "text": image_prompt},
        ]),
    ],
    tokenizer=vision_processor)
vision_prompt = vision_prompt_template.format_prompt()

# Convert document with image extraction
enrichment_options = PdfPipelineOptions(
    generate_picture_images=True,
    images_scale=2.0,
)

converter_enriched = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=enrichment_options)
    }
)

enr_result = converter_enriched.convert(docling_paper)
enr_doc = enr_result.document

# Call Replicate vision model for each picture
for idx, pic in enumerate(enr_doc.pictures):
    image = pic.get_image(enr_doc)
    if image is None:
        continue
    print(f"Describing picture {idx}...")
    description = vision_model.invoke(vision_prompt, image=encode_image(image))
    print(f"  -> {description[:100]}...")
    # Attach description to picture metadata
    if pic.meta is None:
        from docling_core.types.doc.document import PictureMeta
        pic.meta = PictureMeta()
    from docling_core.types.doc.document import DescriptionMetaField
    pic.meta.description = DescriptionMetaField(
        text=description,
        created_by="granite-vision-3.3-2b (Replicate)",
    )

display_enriched_document(enr_doc, model_name="granite-vision-3.3-2b (Replicate)")

## Visual Language Models (VLMs)

A recent feature of Docling is the use of a single shot VLM to process documents. This means that instead the traditional conversion pipeline, we send the document to a custom tuned VLM that can extract all the content directly.


### Using Granite-Docling

[Granite-Docling](https://huggingface.co/ibm-granite/granite-docling-258M) is IBM's compact (258M parameters) vision-language model purpose-built for document conversion. It emits [DocTags](https://github.com/docling-project/docling-core) directly — a structured markup that preserves hierarchy, tables, and layout — so a single forward pass replaces the traditional multi-stage pipeline (layout detection, table structure, OCR).

**Note:** This section runs the model locally on the Colab runtime. For best performance, select a GPU runtime: **Runtime > Change runtime type > T4 GPU**.

In [ ]:
from docling.datamodel import vlm_model_specs
from docling.datamodel.pipeline_options import VlmPipelineOptions
from docling.pipeline.vlm_pipeline import VlmPipeline
from docling.utils.accelerator_utils import decide_device
from docling.datamodel.accelerator_options import AcceleratorDevice

# Use the Transformers build — works on Colab's CPU and GPU runtimes
vlm_options = vlm_model_specs.GRANITEDOCLING_TRANSFORMERS

# Report the hardware docling will actually run Granite-Docling on
_device = decide_device(AcceleratorDevice.AUTO.value, vlm_options.supported_devices)
print(f"Granite-Docling running on {_device} ({vlm_options.inference_framework.value})")
if _device == "cpu":
    print("   No GPU detected — conversion will be slow.")
    print("   In Colab: Runtime → Change runtime type → T4 GPU")

vlm_pipeline_options = VlmPipelineOptions(
    force_backend_text=False,
    vlm_options=vlm_options,
)

converter_vlm = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=vlm_pipeline_options,
            pipeline_cls=VlmPipeline,
        )
    }
)

vlm_result = converter_vlm.convert(docling_paper)
vlm_doc = vlm_result.document

In [ ]:
vlm_md_out = vlm_doc.export_to_markdown()

# printing an excerpt
print(f"{vlm_md_out[:2000]}...")

## Summary and Next Steps

### What You've Accomplished in Lab 1

Congratulations! You've mastered the critical first step in document AI:

- Basic and advanced document conversion with visual feedback
- Multiple export formats with visualization options
- Table and image extraction with visual verification
- Enrichment models via Replicate and VLMs

### Your Journey Continues

You're now ready for:
- **Lab 2**: Learn intelligent chunking strategies to optimize for retrieval
- **Lab 3**: Build a complete multimodal RAG system with visual grounding

### Next Steps

1. **Practice with Your Documents**
   - Try different document types
   - Experiment with configuration options
   - Validate extraction quality

2. **Proceed Lab 2**
   - Gather a collection of documents
   - Think about how you'd want to chunk them
   - Consider what makes a good chunk for retrieval

Docling provides a powerful, flexible framework for document processing that can be adapted to many use cases. The modular architecture allows you to enable only the features you need, while the visualization capabilities ensure transparency and accuracy in your document processing pipeline.

For more information:
- GitHub: https://github.com/docling-project/docling
- Documentation: https://docling-project.github.io/docling/
- Technical Report: https://arxiv.org/abs/2408.09869
- Examples: https://github.com/docling-project/docling/tree/main/examples